# 07 — Backtesting (Separate Run Per Jump Definition)

Strategy S1: Spot momentum — go long/short on BTC at T+1 after a jump fires.
Test each of D1, D2, D3, D4 with holding periods [10, 30, 60, 240] minutes.

Saves all results to `data/processed/backtest_results.parquet` for comparison in notebook 08.

In [ ]:
import sys
sys.path.append('..')

import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.backtest import run_backtest, train_val_test_split, walk_forward
from src.metrics import full_report, passes_minimum_bars
from src.plots import plot_equity_curve, plot_drawdown, plot_rolling_sharpe, plot_monthly_returns_heatmap, plot_gross_vs_net

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

raw_dir  = Path('../data/raw')
proc_dir = Path('../data/processed')
bt_cfg   = cfg['backtest']

# Load BTC returns
btc = pd.read_parquet(raw_dir / 'crypto' / 'crypto_1m_BTCUSDT.parquet')
btc = btc.set_index('timestamp_utc').sort_index()
btc_ret = btc['log_ret']

DEFINITIONS = ['D1', 'D2', 'D3', 'D4']
HOLDING_PERIODS = bt_cfg['holding_periods']  # [10, 30, 60, 240]

signals_dict = {}
for defn in DEFINITIONS:
    path = proc_dir / f'jump_signals_{defn}.parquet'
    if path.exists():
        signals_dict[defn] = pd.read_parquet(path)

print('Config:', bt_cfg)
print('Definitions loaded:', list(signals_dict.keys()))

## Helper: Build Pooled Signal

In [ ]:
def pool_signals(sig_df: pd.DataFrame, btc_ret: pd.Series) -> pd.Series:
    """Stack all market signals; at any minute where multiple markets fire, take largest magnitude."""
    combined = sig_df.stack()
    combined = combined[combined != 0]
    if combined.empty:
        return pd.Series(0.0, index=btc_ret.index)
    flat = combined.groupby(level=0).apply(lambda x: x.iloc[x.abs().argmax()])
    return flat.reindex(btc_ret.index).fillna(0)

## Run Backtests — All Definitions × All Holding Periods

In [ ]:
all_reports = []  # collect for comparison table

for defn, sig_df in signals_dict.items():
    signal = pool_signals(sig_df, btc_ret)
    n_events = int((signal != 0).sum())
    print(f'\n{"="*50}')
    print(f'{defn}: {n_events} total events')

    if n_events < 10:
        print(f'  Too few events, skipping {defn}')
        continue

    # Split into train / validation / test
    train_sig, val_sig, test_sig = train_val_test_split(signal, train_frac=0.6, val_frac=0.2)
    print(f'  Train events: {int((train_sig!=0).sum())}, '
          f'Val: {int((val_sig!=0).sum())}, '
          f'Test: {int((test_sig!=0).sum())}')

    for H in HOLDING_PERIODS:
        label = f'{defn}_H{H}'

        # Run on validation set to find best H
        val_bt = run_backtest(
            val_sig, btc_ret, holding_period=H,
            commission_rt=bt_cfg['commission_rt'],
            slippage=bt_cfg['slippage'],
            stop_loss_vol_mult=bt_cfg['stop_loss_vol_mult'],
        )
        val_report = full_report(val_bt['net_ret'], gross_returns=val_bt['gross_ret'],
                                  benchmark_returns=btc_ret, label=f'{label}_VAL')

        all_reports.append({
            'definition': defn,
            'holding_period': H,
            'split': 'val',
            **val_report.drop('label').to_dict()
        })

    defn_val_reports = [r for r in all_reports if r['definition'] == defn and r['split'] == 'val']
    best_val = max(
        defn_val_reports,
        key=lambda r: -np.inf if pd.isna(r.get('sharpe')) else r.get('sharpe'),
    )
    print(f'  → Best H on validation set: {best_val["holding_period"]}')

## Best H per Definition → Run on TEST SET (once)

In [ ]:
# Find best H per definition based on validation Sharpe
reports_df = pd.DataFrame(all_reports)
val_reports = reports_df[reports_df['split'] == 'val']

test_reports = []

for defn in signals_dict:
    defn_val = val_reports[val_reports['definition'] == defn]
    if defn_val.empty:
        continue

    best_H = defn_val.sort_values('sharpe', ascending=False).iloc[0]['holding_period']
    print(f'{defn}: best H on validation = {int(best_H)} minutes')

    sig_df = signals_dict[defn]
    signal = pool_signals(sig_df, btc_ret)
    _, _, test_sig = train_val_test_split(signal)

    if int((test_sig != 0).sum()) < 5:
        print(f'  {defn}: fewer than 5 test events, skipping')
        continue

    test_bt = run_backtest(
        test_sig, btc_ret, holding_period=int(best_H),
        commission_rt=bt_cfg['commission_rt'],
        slippage=bt_cfg['slippage'],
        stop_loss_vol_mult=bt_cfg['stop_loss_vol_mult'],
    )

    label = f'{defn}_H{int(best_H)}_TEST'
    test_report = full_report(
        test_bt['net_ret'], gross_returns=test_bt['gross_ret'],
        benchmark_returns=btc_ret, label=label
    )

    print(f'  Minimum bars check:')
    passed = passes_minimum_bars(test_report)

    test_reports.append({'definition': defn, 'best_H': int(best_H), 'split': 'test',
                         **test_report.drop('label').to_dict()})

    # Plots
    fig = plot_equity_curve(test_bt['net_ret'], btc_ret, title=f'Equity Curve: {label}')
    plt.show()
    plt.close()

    fig = plot_drawdown(test_bt['net_ret'], title=f'Drawdown: {label}')
    plt.show()
    plt.close()

    fig = plot_gross_vs_net(test_bt['gross_ret'], test_bt['net_ret'], title=f'Cost Drag: {label}')
    plt.show()
    plt.close()

In [ ]:
# Save all reports for comparison notebook
combined_reports = pd.DataFrame(all_reports + test_reports)
combined_reports.to_parquet(proc_dir / 'backtest_results.parquet', index=False)
print('Saved backtest_results.parquet')
print(combined_reports[combined_reports['split']=='test'][['definition','best_H','sharpe','max_drawdown_pct','n_trades']].to_string(index=False))

## Walk-Forward Validation (Best Definition)

In [ ]:
if test_reports:
    best_test = max(test_reports, key=lambda r: r.get('sharpe', -np.inf))
    best_defn = best_test['definition']
    best_H    = best_test['best_H']
    print(f'Walk-forward for best: {best_defn}, H={best_H}')

    signal = pool_signals(signals_dict[best_defn], btc_ret)
    wf_results = walk_forward(
        signal, btc_ret,
        holding_period=best_H,
        n_windows=5,
        train_months=6,
        test_months=2,
        gap_days=7,
        commission_rt=bt_cfg['commission_rt'],
        slippage=bt_cfg['slippage'],
    )

    print(f'\nWalk-forward Sharpe per window:')
    for i, wf_bt in enumerate(wf_results):
        wf_sh = wf_bt['net_ret'].mean() / wf_bt['net_ret'].std() * np.sqrt(525_600)
        print(f'  Window {i+1}: Sharpe={wf_sh:.3f}, trades={int((wf_bt["net_ret"]!=0).sum())}')